In [19]:
!pip install keras-tuner --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.2 MB/s eta 0:00:00


In [58]:
import tensorflow as tf
import numpy as np
import pandas as pd
import keras_tuner as kt

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

X, y = fetch_california_housing(return_X_y=True)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, random_state=42, test_size=0.2, shuffle=True)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, random_state=42, test_size=0.1, shuffle=True)


In [26]:
class OneCycleScheduler(tf.keras.callbacks.Callback):
    def __init__(
        self, max_lr, total_iters, pct_start=0.3, div_factor=10, final_div_factor=1e4):
        super().__init__()

        self.max_lr = max_lr
        self.total_iters = total_iters
        self.pct_start = pct_start

        self.start_lr = max_lr / div_factor
        self.final_lr = max_lr / final_div_factor

        self.step = 0
        self.up_steps = int(total_iters * pct_start)
        self.down_steps = total_iters - self.up_steps

    def on_train_begin(self, logs=None):
        self.model.optimizer.learning_rate.assign(self.start_lr)

    def on_batch_begin(self, batch, logs=None):
        if self.step < self.up_steps:
            lr = self.start_lr + (self.max_lr - self.start_lr) * (self.step / self.up_steps)
        else:
            k = self.step - self.up_steps
            lr = self.max_lr - (self.max_lr - self.final_lr) * (k / self.down_steps)

        self.model.optimizer.learning_rate.assign(lr)
        self.step += 1

In [16]:
norm = tf.keras.layers.Normalization()
norm.adapt(X_train)

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(X_train.shape[1:]),
    norm,
    tf.keras.layers.Dense(300, activation="relu", kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu", kernel_initializer="he_normal"),
    tf.keras.layers.Dense(1)
])

import math
batch_size = 32
n_epochs = 30
n_steps = n_epochs * math.ceil(len(X_train) / batch_size)

optimizer = tf.keras.optimizers.SGD()
model.compile(loss='mse', optimizer=optimizer, metrics=['RootMeanSquaredError'])
hist = model.fit(X_train, y_train, epochs=n_epochs, validation_data=(X_val, y_val),
          callbacks=[OneCycleScheduler(0.01, n_steps)])

Epoch 1/30
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - RootMeanSquaredError: 1.3033 - loss: 1.7387 - val_RootMeanSquaredError: 0.8506 - val_loss: 0.7235
Epoch 2/30
465/465 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - RootMeanSquaredError: 2.1051 - loss: 6.7272 - val_RootMeanSquaredError: 1.2156 - val_loss: 1.4777
Epoch 3/30
465/465 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - RootMeanSquaredError: 0.8347 - loss: 0.6980 - val_RootMeanSquaredError: 0.7661 - val_loss: 0.5868
Epoch 4/30
465/465 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - RootMeanSquaredError: 0.7159 - loss: 0.5131 - val_RootMeanSquaredError: 0.7120 - val_loss: 0.5069
Epoch 5/30
465/465 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - RootMeanSquaredError: 0.6740 - loss: 0.4544 - val_RootMeanSquaredError: 0.7766 - val_loss: 0.6032
Epoch 6/30
465/465 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - RootMeanSquaredError: 0.7868 - loss: 0.6274 - val_RootMeanSquaredError: 0.6817 - val_loss: 0.4647
Epoch 7/30
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - RootMeanSquaredError: 0.6365 - los

In [17]:
model.evaluate(X_test, y_test)

129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - RootMeanSquaredError: 0.5549 - loss: 0.3082


[0.31478968262672424, 0.5610612034797668]

## Task 6

да се тества класификатор на fasion MINST

    да се ползва grid search
    със стандартен SGD със всички видове параметри
    със всички други optimizer-и
    със batch normalization
    със различните learning rate schedule имплементации
    комбинация от momentum optimization и learning scheduling

да се сравнят времената на обучение и производителността на обучените модели

In [52]:
fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

X_train_full, X_train, X_valid, X_test = X_train_full / 255., X_train / 255., X_valid / 255., X_test / 255.
X_train_full = X_train_full.reshape(-1, 28, 28, 1)
X_valid = X_valid.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

callbacks

In [53]:
class PowerLR(tf.keras.callbacks.Callback):
    def __init__(self, lr0, k=1e-4, power=1.0):
        self.lr0 = lr0
        self.k = k
        self.power = power
        self.step = 0

    def on_batch_begin(self, batch, logs=None):
        lr = self.lr0 * (1 + self.k * self.step) ** (-self.power)
        self.model.optimizer.learning_rate.assign(lr)
        self.step += 1

class ExponentialLR(tf.keras.callbacks.Callback):
    def __init__(self, lr0, decay_rate=0.99):
        self.lr0 = lr0
        self.decay_rate = decay_rate
        self.step = 0

    def on_batch_begin(self, batch, logs=None):
        lr = self.lr0 * (self.decay_rate ** self.step)
        self.model.optimizer.learning_rate.assign(lr)
        self.step += 1


class PiecewiseConstantLR(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if epoch < 5:
          lr = 0.01
        elif epoch < 15:
          lr = 0.005
        else:
          lr = 0.001

        self.model.optimizer.learning_rate.assign(lr)

In [54]:
import math
batch_size = 32
n_epochs = 20
n_steps = n_epochs * math.ceil(len(X_train) / batch_size)

def build_model(hp):
    n_hidden = hp.Int("n_hidden", min_value=0, max_value=8, default=2)
    n_neurons = hp.Int("n_neurons", min_value=16, max_value=100)
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate,
                                        momentum=hp.Float("momentum", 0.0, 0.95, step=0.05),
                                        nesterov=hp.Boolean("nesterov"))

    model = tf.keras.Sequential()

    model.add(tf.keras.layers.Input([28, 28, 1]))
    model.add(tf.keras.layers.Flatten())

    batch_norm_flag = hp.Boolean("batch_norm")

    if batch_norm_flag:
        for _ in range(n_hidden):
            model.add(tf.keras.layers.Dense(n_neurons,
                                            kernel_initializer='he_normal'))
            model.add(tf.keras.layers.BatchNormalization())
            model.add(tf.keras.layers.ReLU())
    else:
        for _ in range(n_hidden):
            model.add(tf.keras.layers.Dense(n_neurons, activation="relu",
                                            kernel_initializer='he_normal'))

    model.add(tf.keras.layers.Dense(10, activation="softmax"))
    model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                                                metrics=["accuracy"])
    return model

In [55]:
class MyClassificationHyperModel(kt.HyperModel):
  def build(self, hp):
    return build_model(hp)

  def fit(self, hp, model, X, y, **kwargs):
    lr_type = hp.Choice("lr_schedule", ["constant", "power", "exponential", "performance", "onecycle"])

    base_lr = hp.Float("base_lr", 1e-4, 5e-2, sampling="log")
    callbacks = [tf.keras.callbacks.EarlyStopping(patience=4)]
    match lr_type:
        case "power":
          callbacks.append(PowerLR(base_lr))
        case "exponential":
          callbacks.append(ExponentialLR(base_lr))
        case "onecycle":
          callbacks.append(OneCycleScheduler(0.1, n_steps))
        case "performance":
          callbacks.append(tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2))

    if "callbacks" in kwargs:
        kwargs["callbacks"] = kwargs["callbacks"] + callbacks
    else:
        kwargs["callbacks"] = callbacks

    return model.fit(X, y, **kwargs)

In [56]:
tuner = kt.Hyperband(
    MyClassificationHyperModel(),
    objective="val_accuracy",
    max_epochs=20,
    factor=3,
    directory="fashion_mnist",
    project_name="schedulers",
    hyperband_iterations=2,
    seed=42,
)

tuner.search(
    X_train_full, y_train_full,
    validation_data=(X_valid, y_valid),
    epochs=20,
    batch_size=32,
)

Trial 56 Complete [00h 02m 06s]
val_accuracy: 0.551800012588501

Best val_accuracy So Far: 0.9211999773979187
Total elapsed time: 00h 57m 09s


In [64]:
for trial in tuner.oracle.get_best_trials(num_trials=5):
    print("Trial ID:", trial.trial_id, "-> Score:", trial.score)
    print(pd.Series(trial.hyperparameters.values))

Trial ID: 0052 -> Score: 0.9211999773979187
n_hidden                      8
n_neurons                    93
learning_rate          0.003424
momentum                    0.7
nesterov                   True
batch_norm                 True
lr_schedule            constant
base_lr                0.000124
tuner/epochs                 20
tuner/initial_epoch           0
tuner/bracket                 0
tuner/round                   0
dtype: object
Trial ID: 0023 -> Score: 0.9154000282287598
n_hidden                      3
n_neurons                    63
learning_rate          0.004264
momentum                   0.55
nesterov                   True
batch_norm                 True
lr_schedule            constant
base_lr                0.000849
tuner/epochs                 20
tuner/initial_epoch           0
tuner/bracket                 0
tuner/round                   0
dtype: object
Trial ID: 0050 -> Score: 0.9143999814987183
n_hidden                      5
n_neurons                    86
learning

In [66]:
best_model = tuner.get_best_models(1)[0]
best_model.evaluate(X_test, y_test)

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'SGD', because it has 2 variables whereas the saved optimizer has 36 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8843 - loss: 0.3257


[0.32233309745788574, 0.8853999972343445]